# Projet Data Quality - BoursoBank
## Contexte métier
BoursoBank est une banque 100% digitale avec 8 millions de clients.
Toutes les operations passent par le Systeme d'Information (SI).
Si les donnees du SI sont mauvaises, les decisions business et reglementaires sont faussees.

## Objectif de ce notebook
Explorer le dataset de transactions pour :
- Comprendre la structure des donnees
- Identifier les premieres anomalies visibles
- Poser les bases de nos regles de gestion

## Dataset
Source : Synthetic data from a financial payment system (Edgar Lopez-Rojas)
Contexte : Transactions bancaires europeennes synthetiques

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

# Chargement du dataset
df = pd.read_csv("../data/bs140513_032310.csv")

# Premier regard
print(f"Dimensions : {df.shape[0]:,} lignes x {df.shape[1]} colonnes")
print(f"\nColonnes : {list(df.columns)}")
print(f"\nTypes de données :")
print(df.dtypes)

Dimensions : 594,643 lignes x 10 colonnes

Colonnes : ['step', 'customer', 'age', 'gender', 'zipcodeOri', 'merchant', 'zipMerchant', 'category', 'amount', 'fraud']

Types de données :
step             int64
customer           str
age                str
gender             str
zipcodeOri         str
merchant           str
zipMerchant        str
category           str
amount         float64
fraud            int64
dtype: object


In [3]:
df.head(10)

,step,customer,age,gender,zipcodeOri,merchant,zipMerchant,category,amount,fraud
0,0,'C1093826151','4','M','28007','M348934600','28007','es_transportation',4.55,0
1,0,'C352968107','2','M','28007','M348934600','28007','es_transportation',39.68,0
2,0,'C2054744914','4','F','28007','M1823072687','28007','es_transportation',26.89,0
3,0,'C1760612790','3','M','28007','M348934600','28007','es_transportation',17.25,0
4,0,'C757503768','5','M','28007','M348934600','28007','es_transportation',35.72,0
5,0,'C1315400589','3','F','28007','M348934600','28007','es_transportation',25.81,0
6,0,'C765155274','1','F','28007','M348934600','28007','es_transportation',9.10,0
7,0,'C202531238','4','F','28007','M348934600','28007','es_transportation',21.17,0
8,0,'C105845174','3','M','28007','M348934600','28007','es_transportation',32.40,0
9,0,'C39858251','5','F','28007','M348934600','28007','es_transportation',35.40,0


In [5]:
# Combien de fraudes dans le dataset ?
# value_counts() compte combien de fois chaque valeur apparait


print("=== Repartition fraude vs legitime ===")
print(df["fraud"].value_counts())

print(f"\nPourcentage de fraudes : {df['fraud'].mean() * 100:.2f}%")

=== Repartition fraude vs legitime ===
fraud
0    587443
1      7200
Name: count, dtype: int64

Pourcentage de fraudes : 1.21%


In [6]:
fraude_par_categorie = df.groupby("category")["fraud"].mean() * 100
fraude_par_categorie = fraude_par_categorie.sort_values(ascending=False)

print("=== Taux de fraude par categorie ===")
print(fraude_par_categorie.round(2))

=== Taux de fraude par categorie ===
category
'es_leisure'               94.99
'es_travel'                79.40
'es_sportsandtoys'         49.53
'es_hotelservices'         31.42
'es_otherservices'         25.00
'es_home'                  15.21
'es_health'                10.51
'es_tech'                   6.67
'es_wellnessandbeauty'      4.76
'es_hyper'                  4.59
'es_barsandrestaurants'     1.88
'es_fashion'                1.80
'es_contents'               0.00
'es_food'                   0.00
'es_transportation'         0.00
Name: fraud, dtype: float64


In [7]:
# Nombre de transactions par categorie
# On veut savoir si es_leisure a beaucoup ou peu de transactions

transactions_par_categorie = df.groupby("category")["fraud"].agg(["count", "sum"])
transactions_par_categorie.columns = ["total_transactions", "total_fraudes"]
transactions_par_categorie["taux_fraude"] = (transactions_par_categorie["total_fraudes"] / transactions_par_categorie["total_transactions"] * 100).round(2)
transactions_par_categorie = transactions_par_categorie.sort_values("taux_fraude", ascending=False)

print(transactions_par_categorie)

                         total_transactions  total_fraudes  taux_fraude
category                                                               
'es_leisure'                            499            474        94.99
'es_travel'                             728            578        79.40
'es_sportsandtoys'                     4002           1982        49.53
'es_hotelservices'                     1744            548        31.42
'es_otherservices'                      912            228        25.00
'es_home'                              1986            302        15.21
'es_health'                           16133           1696        10.51
'es_tech'                              2370            158         6.67
'es_wellnessandbeauty'                15086            718         4.76
'es_hyper'                             6098            280         4.59
'es_barsandrestaurants'                6373            120         1.88
'es_fashion'                           6454            116      

In [8]:
# Montant moyen par categorie
# On veut verifier si les categories fraudees ont des montants plus eleves

montant_par_categorie = df.groupby("category")["amount"].mean().round(2)
montant_par_categorie = montant_par_categorie.sort_values(ascending=False)

print("=== Montant moyen par categorie ===")
print(montant_par_categorie)

=== Montant moyen par categorie ===
category
'es_travel'                2250.41
'es_leisure'                288.91
'es_sportsandtoys'          215.72
'es_hotelservices'          205.61
'es_home'                   165.67
'es_otherservices'          135.88
'es_health'                 135.62
'es_tech'                   120.95
'es_fashion'                 65.67
'es_wellnessandbeauty'       65.51
'es_hyper'                   45.97
'es_contents'                44.55
'es_barsandrestaurants'      43.46
'es_food'                    37.07
'es_transportation'          26.96
Name: amount, dtype: float64
